# RAG-ассистент поддержки онлайн-кинотеатра «Фильмоскоп»

**Репозиторий проекта:** [github.com/enterlina/filmoskop-rag](https://github.com/enterlina/filmoskop-rag)

## Описание задачи

Необходимо разработать прототип RAG-ассистента службы поддержки онлайн-кинотеатра «Фильмоскоп». Система должна находить релевантные разделы базы знаний, формировать краткий ответ только на основании найденного контекста и корректно отказываться от ответа, если нужной информации в базе нет.

В исходных данных находятся 181 документ и 66 тестовых запросов четырёх типов: `exact` — вопросы с точными названиями, числами и кодами ошибок; `semantic` — разговорные переформулировки; `multi` — вопросы, для которых требуется несколько документов; `gap` — вопросы без ответа в базе знаний.

## Цели проекта

1. Провести аудит корпуса, исправить проблемы кодировки и удалить архивные, служебные и дублирующие документы.
2. Построить воспроизводимый retrieval-пайплайн и достичь `recall@5 ≥ 0.85`.
3. Сравнить dense retrieval, cross-encoder reranking, BM25, Query Expansion и HyDE.
4. Сравнить генераторы Nemotron и GPT-OSS 20B по faithfulness, relevance и correctness.
5. Реализовать согласованный отказ для `gap`-запросов и сохранить промежуточные результаты в чекпойнты.
6. Проанализировать ошибки и определить, готова ли система к пилотному использованию.

## Архитектура решения

Финальный маршрут поиска: очистка и paragraph-aware чанкинг → multilingual E5 embeddings → BM25+dense RRF для exact-запросов или dense retrieval для остальных → top-20 кандидатов → BGE cross-encoder reranker → top-5 контекстов → GPT-OSS 20B через NVIDIA API.

Компоненты вызываются напрямую, без LangChain, чтобы преобразования, ранжирование и метрики оставались прозрачными. Плотный поиск реализован точным скалярным произведением L2-нормированных векторов, функционально эквивалентным `IndexFlatIP`. Retrieval-метрики считаются по 56 запросам с ответом; 10 `gap`-запросов проверяются отдельно на этапе генерации.

Ноутбук следует запускать из каталога `ypraktikum_task2/filmoskop_project`. Секретный NVIDIA API key вводится через скрытый запрос и не сохраняется в файле.

In [51]:
from __future__ import annotations

import hashlib, json, math, os, random, re, time, unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, CrossEncoder

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_URL = (
    "https://code.s3.yandex.net/ai-engineer/"
    "%D0%98%D1%82%D0%BE%D0%B3%D0%BE%D0%B2%D1%8B%D0%B9_"
    "%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82_"
    "%D1%81%D0%BF%D1%80%D0%B8%D0%BD%D1%822/filmoskop_kb.json"
)
DATA_PATH = Path("filmoskop_kb.json")
if not DATA_PATH.exists():
    print("Скачиваем официальный датасет Яндекс Практикума...")
    urlretrieve(DATA_URL, DATA_PATH)

EMBED_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
# Большой реранкер принудительно выполняется на CPU: одновременная загрузка
# E5, реранкера и локальной Qwen может переполнить общую Metal-память macOS.
RERANK_DEVICE = "cpu"
REFUSAL = "В базе знаний нет информации для ответа на этот вопрос."
TOP_K = 5
CHECKPOINT_VERSION = 3  # v3: фильтрация архива/служебных документов и дедупликация
CHECKPOINT_DIR = Path("filmoskop_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def cache_key(*objects):
    "Короткий устойчивый ключ: старый кэш не подмешивается к новым данным/настройкам."
    payload = json.dumps(objects, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]

def save_json_atomic(path, value):
    "Атомарно сохраняет JSON, не оставляя битый файл при остановке процесса."
    path = Path(path); temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)
raw_docs, queries = data["docs"], data["queries"]
print(f"Разделов: {len(raw_docs)} | запросов: {len(queries)}")
print("Типы:", Counter(q["kind"] for q in queries))
print("Чекпоинты:", CHECKPOINT_DIR.resolve())

Разделов: 181 | запросов: 66
Типы: Counter({'semantic': 25, 'multi': 16, 'exact': 15, 'gap': 10})
Чекпоинты: /Users/elenadalbonidarocha/Documents/Projects_yan/ai_assistant_free_track/ypraktikum_task2/filmoskop_project/filmoskop_checkpoints


## Этап 1. Аудит и подготовка корпуса

Проверяем длины, долю символов замены/управляющих символов, остатки HTML и
дубли после нормализации содержимого. HTML разбирается парсером, Markdown-маркеры
убираются без удаления содержимого, Unicode приводится к NFC. Архивные и явно
служебные материалы исключаются по меткам заголовка. Контентные дубли ищутся
по сходству очищенного текста, а не по совпадению названий; из группы дублей
сохраняется наиболее новая запись с максимальным `doc_id`.

In [52]:
def quality_row(doc):
    "Возвращает простые диагностические признаки сырого раздела."
    text = doc["raw"]
    n = max(len(text), 1)
    bad = sum(ch == "\ufffd" or unicodedata.category(ch).startswith("C")
              for ch in text)
    return {"doc_id": doc["doc_id"], "format": doc["format"],
            "category": doc["category"], "length": len(text),
            "junk_ratio": bad / n,
            "has_markup": bool(re.search(r"<[^>]+>|^#{1,6}\s", text, re.M)),
            "has_mojibake": bool(re.search(r"(?:Ð.|Ñ.|â.|Â.){2,}", text))}

def repair_mojibake(text):
    "Построчно исправляет UTF-8, ошибочно декодированный как Latin-1/CP1252."
    suspicious = lambda s: sum(s.count(ch) for ch in ("Ð", "Ñ", "Â", "Ã", "â", "�"))
    def repair_part(part):
        candidates = [part]
        for encoding in ("latin1", "cp1252"):
            try:
                candidates.append(part.encode(encoding).decode("utf-8"))
            except (UnicodeEncodeError, UnicodeDecodeError):
                pass
        return min(candidates, key=lambda s: (suspicious(s), -sum("а" <= ch.lower() <= "я" for ch in s)))
    return "".join(repair_part(line) for line in text.splitlines(keepends=True))

def clean_text(doc):
    "Очищает HTML/Markdown и нормализует Unicode и пробелы."
    text = doc["raw"]
    if doc["format"] == "html":
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = soup.get_text("\n")
    text = repair_mojibake(text)
    text = unicodedata.normalize("NFC", text).replace("\xa0", " ")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.M)
    text = re.sub(r"^\s*[-*]\s+", "", text, flags=re.M)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

ARCHIVE_RE = re.compile(r"\(архив(?:[^)]*)?\)", re.I)
SERVICE_RE = re.compile(
    r"служебн|шаблон ответа|регламент|протокол встречи|конфигурац|"
    r"вакансия|черновик|проверка в[её]рстки|схема разделов|"
    r"раздел перенес[её]н|статья обновляется|раздел в разработке", re.I)

def content_signature(text):
    "Нормализация содержимого для поиска точных и почти точных дублей."
    return re.sub(r"[^a-zа-яё0-9]+", "", text.lower())

def filter_and_deduplicate(documents, similarity_threshold=.92):
    "Удаляет архив/служебные записи и почти полные контентные дубли."
    excluded, active = [], []
    for d in documents:
        reason = ("archive" if ARCHIVE_RE.search(d["title"]) else
                  "service" if SERVICE_RE.search(d["title"]) else None)
        (excluded if reason else active).append(({**d, "exclude_reason": reason}) if reason else d)
    kept, duplicates = [], []
    # Новая запись сравнивается первой, поэтому при дубле сохраняется max(doc_id).
    for d in sorted(active, key=lambda x: x["doc_id"], reverse=True):
        signature = content_signature(d["text"])
        duplicate_of = next((x for x in kept if min(len(signature), len(x["_signature"])) >= 50
                             and SequenceMatcher(None, signature, x["_signature"]).ratio()
                             >= similarity_threshold), None)
        if duplicate_of:
            duplicates.append({**d, "duplicate_of": duplicate_of["doc_id"]})
        else:
            kept.append({**d, "_signature": signature})
    return ([{k: v for k, v in d.items() if k != "_signature"}
             for d in sorted(kept, key=lambda x: x["doc_id"])], excluded, duplicates)

audit_before = pd.DataFrame(quality_row(d) for d in raw_docs)
cleaned_docs = []
for d in raw_docs:
    cleaned = clean_text(d)
    if cleaned:
        cleaned_docs.append({**d, "text": cleaned})
docs, excluded_docs, duplicate_docs = filter_and_deduplicate(cleaned_docs)
audit_after = pd.DataFrame(quality_row({**d, "raw": d["text"], "format": "text"})
                           for d in docs)
display(audit_before.groupby("format")["length"].agg(["count", "min", "median", "max"]))
print("Разметка: было", audit_before.has_markup.sum(), "→ стало", audit_after.has_markup.sum())
print("Mojibake: было", audit_before.has_mojibake.sum(), "→ стало", audit_after.has_mojibake.sum())
print("Исключено архивных:", sum(d["exclude_reason"] == "archive" for d in excluded_docs),
      "| служебных:", sum(d["exclude_reason"] == "service" for d in excluded_docs),
      "| контентных дублей:", len(duplicate_docs), "| осталось:", len(docs))
display(pd.DataFrame([{"doc_id": d["doc_id"], "title": d["title"],
                       "reason": d.get("exclude_reason", "duplicate"),
                       "duplicate_of": d.get("duplicate_of")}
                      for d in [*excluded_docs, *duplicate_docs]]))
for d in raw_docs:
    if d["format"] in {"html", "markdown"}:
        print("ДО:", d["raw"][:220].replace("\n", " "))
        print("ПОСЛЕ:", clean_text(d)[:220].replace("\n", " "))
        break

,count,min,median,max
format,,,,
html,21,42,570.0,2818
markdown,28,420,802.5,3057
text,132,14,659.0,3017


Разметка: было 49 → стало 0
Mojibake: было 16 → стало 0
Исключено архивных: 21 | служебных: 13 | контентных дублей: 7 | осталось: 140


,doc_id,title,reason,duplicate_of
0,10,Вход на телевизоре по коду (архив),archive,NaN
1,17,Скорость интернета для просмотра (архив),archive,NaN
2,25,Тарифы «Фильмоскопа» (архив),archive,NaN
3,27,Совместный просмотр (архив),archive,NaN
4,28,Объёмный звук 5.1 (архив),archive,NaN
5,33,Привязанные устройства (архив),archive,NaN
6,36,Регламент обработки обращений об ошибках,service,NaN
7,45,Требования для просмотра в 4K (архив),archive,NaN
8,52,Раздел справки (архив),archive,NaN
9,53,Раздел перенесён,service,NaN


ДО: # Безопасность аккаунта: общие советы  Несколько привычек, которые защищают аккаунт лучше любых настроек:  используйте уникальный пароль, который не повторяется на других сайтах; не вводите данные аккаунта на сторонних с
ПОСЛЕ: Безопасность аккаунта: общие советы  Несколько привычек, которые защищают аккаунт лучше любых настроек:  используйте уникальный пароль, который не повторяется на других сайтах; не вводите данные аккаунта на сторонних сай


**Вывод этапа 1.** Очистка восстанавливает полезный текст разметки и mojibake.
До индексации исключаются пустые, архивные и явно служебные разделы. Почти
полные дубли определяются по сходству нормализованного содержимого с порогом
0.92; сохраняется более новая запись. Таблица выше делает каждое исключение
проверяемым и не позволяет скрыть полезный документ молча.

## Этап 2. Чанкинг и эмбеддинги

Сравниваются две стратегии: целый раздел и абзацно-ориентированные чанки до
1200 символов. Во второй стратегии заголовок повторяется в каждом чанке — это
сохраняет подлежащее и контекст. Граница выбирается по абзацу, поэтому факт не
режется посередине предложения. На маленьких документах стратегии совпадают.

In [53]:
def make_chunks(documents, strategy="paragraph", max_chars=1200):
    "Создаёт чанки и сохраняет метаданные исходного раздела."
    result = []
    for d in documents:
        title, body = d["title"].strip(), d["text"].strip()
        if strategy == "document":
            parts = [body]
        else:
            paragraphs = [p.strip() for p in re.split(r"\n+", body) if p.strip()]
            parts, current = [], ""
            for p in paragraphs:
                candidate = f"{current}\n{p}".strip()
                if current and len(candidate) > max_chars:
                    parts.append(current); current = p
                else:
                    current = candidate
            if current: parts.append(current)
        for part_no, part in enumerate(parts):
            result.append({"chunk_id": len(result), "doc_id": d["doc_id"],
                           "title": title, "category": d["category"],
                           "text": f"{title}\n{part}", "part_no": part_no})
    return result

chunks_document = make_chunks(docs, "document")
chunks_paragraph = make_chunks(docs, "paragraph")
print("Целый раздел:", len(chunks_document), "| по абзацам:", len(chunks_paragraph))
display(pd.DataFrame({
    "strategy": ["document", "paragraph"],
    "chunks": [len(chunks_document), len(chunks_paragraph)],
    "median_chars": [np.median([len(x["text"]) for x in chunks_document]),
                     np.median([len(x["text"]) for x in chunks_paragraph])],
    "max_chars": [max(len(x["text"]) for x in chunks_document),
                  max(len(x["text"]) for x in chunks_paragraph)]}))

Целый раздел: 140 | по абзацам: 159


,strategy,chunks,median_chars,max_chars
0,document,140,747.5,2109
1,paragraph,159,716.0,1223


In [54]:
embedder = SentenceTransformer(EMBED_MODEL)

def encode_chunks(chunks):
    "Кодирует E5 passage-префиксом и L2-нормализацией."
    return embedder.encode(["passage: " + c["text"] for c in chunks],
                           normalize_embeddings=True, batch_size=32,
                           show_progress_bar=True)

embedding_key = cache_key(CHECKPOINT_VERSION, EMBED_MODEL,
                          [q["query"] for q in queries],
                          chunks_document, chunks_paragraph)
embedding_path = CHECKPOINT_DIR / f"embeddings_{embedding_key}.npz"
if embedding_path.exists():
    saved = np.load(embedding_path)
    query_vectors = saved["query_vectors"]
    vectors_document = saved["vectors_document"]
    vectors_paragraph = saved["vectors_paragraph"]
    print("Эмбеддинги загружены из чекпоинта:", embedding_path)
else:
    query_vectors = embedder.encode(["query: " + q["query"] for q in queries],
                                    normalize_embeddings=True, batch_size=32,
                                    show_progress_bar=True)
    vectors_document = encode_chunks(chunks_document)
    vectors_paragraph = encode_chunks(chunks_paragraph)
    temporary_embedding_path = embedding_path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary_embedding_path, query_vectors=query_vectors,
                        vectors_document=vectors_document,
                        vectors_paragraph=vectors_paragraph)
    temporary_embedding_path.replace(embedding_path)
    print("Эмбеддинги сохранены:", embedding_path)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6001.56it/s]


Эмбеддинги загружены из чекпоинта: filmoskop_checkpoints/embeddings_30a426d1924633ab.npz


## Этап 3. Поисковая инфраструктура и baseline

Используем exact cosine search: после L2-нормализации это матричное скалярное
произведение, функционально равное FAISS `IndexFlatIP`. Для 181–205 объектов
оно проще и гарантирует точный top-K. Переход на HNSW/IVF имеет смысл при
сотнях тысяч или миллионах чанков после отдельного измерения recall/latency.

In [55]:
def unique_doc_ranking(scores, chunks, k=20, category=None):
    "Переводит ranking чанков в уникальные doc_id с метаданными."
    order = np.argsort(-scores); out, seen = [], set()
    for i in order:
        c = chunks[int(i)]
        if category is not None and c["category"] != category: continue
        if c["doc_id"] in seen: continue
        seen.add(c["doc_id"]); out.append({**c, "score": float(scores[i])})
        if len(out) == k: break
    return out

def metrics_for_rankings(rankings, ks=(1, 3, 5, 10, 20)):
    "Считает section-level precision, recall и MRR без gap-запросов."
    rows = []
    for q, ranking in zip(queries, rankings):
        if q["kind"] == "gap": continue
        rel, ids = set(q["relevant_docs"]), [x["doc_id"] for x in ranking]
        row = {"kind": q["kind"], "query": q["query"],
               "precision@5": len(rel & set(ids[:5])) / 5,
               "recall@5": len(rel & set(ids[:5])) / len(rel),
               "mrr": next((1/(i+1) for i, x in enumerate(ids) if x in rel), 0)}
        row.update({f"recall@{k}": len(rel & set(ids[:k])) / len(rel) for k in ks})
        rows.append(row)
    return pd.DataFrame(rows)

def dense_rankings(chunks, vectors):
    return [unique_doc_ranking(qv @ vectors.T, chunks, 20) for qv in query_vectors]

def lexical_tokens(text):
    return re.findall(r"[a-zа-яё0-9]+(?:[-.][a-zа-яё0-9]+)*", text.lower())

BM25_TOKENS = [lexical_tokens(c["text"]) for c in chunks_paragraph]
BM25_DF = Counter(token for tokens in BM25_TOKENS for token in set(tokens))
BM25_AVGDL = np.mean([len(x) for x in BM25_TOKENS])

def bm25_scores(question, k1=1.5, b=.75):
    "Классический BM25 без внешней зависимости; один score на каждый chunk."
    scores, n_docs = np.zeros(len(BM25_TOKENS)), len(BM25_TOKENS)
    for token in lexical_tokens(question):
        df = BM25_DF.get(token, 0)
        if not df: continue
        idf = math.log(1 + (n_docs - df + .5) / (df + .5))
        for i, tokens in enumerate(BM25_TOKENS):
            tf = tokens.count(token)
            if tf:
                denom = tf + k1 * (1 - b + b * len(tokens) / BM25_AVGDL)
                scores[i] += idf * tf * (k1 + 1) / denom
    return scores

def query_route(question):
    "Маршрутизатор использует только текст запроса, но не тестовую метку kind."
    if re.search(r"[A-ZА-Я]{1,5}[- ]?\d|\d+(?:[.,]\d+)?|[«\"]", question):
        return "exact"
    if re.search(r"\b(?:и|также|одновременно|при переходе|а ещё)\b", question.lower()):
        return "multi"
    return "semantic"

def rrf_fuse(dense_ranking, lexical_ranking, k=20, rrf_k=60):
    "Reciprocal Rank Fusion двух списков с сохранением метаданных chunks."
    fused, representatives = defaultdict(float), {}
    for ranking in (dense_ranking, lexical_ranking):
        for rank, item in enumerate(ranking, 1):
            fused[item["doc_id"]] += 1 / (rrf_k + rank)
            representatives.setdefault(item["doc_id"], item)
    order = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{**representatives[doc_id], "rrf_score": fused[doc_id]} for doc_id in order]

def hybrid_candidates(question, query_vector, k=20):
    dense = unique_doc_ranking(query_vector @ vectors_paragraph.T, chunks_paragraph, 40)
    if query_route(question) != "exact":
        return dense[:k]
    lexical = unique_doc_ranking(bm25_scores(question), chunks_paragraph, 40)
    return rrf_fuse(dense, lexical, k=k)

dense_doc = dense_rankings(chunks_document, vectors_document)
dense_para = dense_rankings(chunks_paragraph, vectors_paragraph)
hybrid_para = [hybrid_candidates(q["query"], qv)
               for q, qv in zip(queries, query_vectors)]
m_doc, m_para = metrics_for_rankings(dense_doc), metrics_for_rankings(dense_para)
m_hybrid = metrics_for_rankings(hybrid_para)
comparison = pd.DataFrame({
    "strategy": ["whole document", "paragraph-aware", "routed BM25 + dense"],
    "recall@5": [m_doc["recall@5"].mean(), m_para["recall@5"].mean(),
                 m_hybrid["recall@5"].mean()],
    "MRR": [m_doc.mrr.mean(), m_para.mrr.mean(), m_hybrid.mrr.mean()]})
display(comparison)
chunks, doc_vectors = chunks_paragraph, vectors_paragraph
baseline_rankings, baseline_metrics = dense_para, m_para
retrieval_rankings, retrieval_metrics = hybrid_para, m_hybrid
print("Выбран routed hybrid: BM25+dense для exact, dense для остальных.")

# Кривая recall@K и пример фильтрации по категории
display(baseline_metrics[["recall@1", "recall@3", "recall@5", "recall@10", "recall@20"]].mean().to_frame("mean"))
example_scores = query_vectors[0] @ doc_vectors.T
display(pd.DataFrame(unique_doc_ranking(example_scores, chunks, 5,
                                        category="Подписка и оплата")))

,strategy,recall@5,MRR
0,whole document,0.785714,0.656817
1,paragraph-aware,0.758929,0.655288
2,routed BM25 + dense,0.767857,0.652907


Выбран routed hybrid: BM25+dense для exact, dense для остальных.


,mean
recall@1,0.410714
recall@3,0.696429
recall@5,0.758929
recall@10,0.946429
recall@20,1.000000


,chunk_id,doc_id,title,category,text,part_no,score
0,96,116,Часто задаваемые вопросы об оплате,Подписка и оплата,Часто задаваемые вопросы об оплате\nЗдесь собр...,0,0.771869
1,133,155,Пробный период,Подписка и оплата,"Пробный период\nСколько времени осталось, видн...",1,0.768957
2,31,39,Экспорт обращений за неделю,Подписка и оплата,Экспорт обращений за неделю\nticket_id;categor...,0,0.767112
3,131,154,"Что делать, если оплата не прошла",Подписка и оплата,"Что делать, если оплата не прошла\nЧто делать,...",0,0.763806
4,28,35,Почему банк отклоняет платёж,Подписка и оплата,Почему банк отклоняет платёж\nРешение о списан...,0,0.760402


**Порог отказа.** Маршрутизатор по тексту относит запрос к exact, semantic или
multi. Для каждого маршрута ниже вычисляется собственный 5-й перцентиль top-1
dense score ответных запросов; gap-разметка в подборе порога не используется.
Код генерации проверяет именно этот route-specific dense threshold. В production
пороги необходимо перенести на отдельную calibration-выборку.

In [56]:
route_score_groups = defaultdict(list)
for q, ranking in zip(queries, baseline_rankings):
    if q["kind"] != "gap":
        route_score_groups[query_route(q["query"])].append(ranking[0]["score"])
ROUTE_THRESHOLDS = {route: float(np.quantile(scores, .05))
                    for route, scores in route_score_groups.items()}
THRESHOLD = min(ROUTE_THRESHOLDS.values())  # обратная совместимость отчёта
display(pd.Series(ROUTE_THRESHOLDS, name="dense threshold").to_frame())

,dense threshold
exact,0.852922
semantic,0.804649
multi,0.830701


## Этап 4. Диагноз и улучшение retrieval

Baseline теряет exact-запросы из-за кодов и редких терминов, semantic — из-за
лексического разрыва, multi — когда второй раздел оказывается ниже top-5.
Поэтому для запросов с кодами, версиями, числами и точными названиями BM25 и
dense-кандидаты объединяются через RRF; semantic-запросы остаются на dense,
чтобы не вносить лексический шум. Полученные top-20 затем передаются
multilingual cross-encoder, который оценивает пару «запрос–текст» целиком.

In [57]:
reranker = CrossEncoder(RERANK_MODEL, device=RERANK_DEVICE)
print(f"Реранкер запущен на {RERANK_DEVICE}: медленнее, но без переполнения Metal-памяти.")

def rerank_all(base_rankings, candidate_k=20):
    "Реранжирует dense-кандидатов cross-encoder моделью."
    result, latencies = [], []
    for q, ranking in zip(queries, base_rankings):
        candidates = ranking[:candidate_k]
        t0 = time.perf_counter()
        scores = reranker.predict([(q["query"], x["text"]) for x in candidates],
                                  batch_size=16, show_progress_bar=False)
        order = np.argsort(-np.asarray(scores))
        result.append([{**candidates[i], "rerank_score": float(scores[i])} for i in order])
        latencies.append(time.perf_counter() - t0)
    return result, latencies

rerank_key = cache_key(CHECKPOINT_VERSION, RERANK_MODEL, RERANK_DEVICE, retrieval_rankings)
rerank_path = CHECKPOINT_DIR / f"reranking_{rerank_key}.json"
if rerank_path.exists():
    cached_rerank = json.loads(rerank_path.read_text(encoding="utf-8"))
    improved_rankings = cached_rerank["rankings"]
    rerank_latency = cached_rerank["latencies"]
    print("Результаты реранжирования загружены из чекпоинта:", rerank_path)
else:
    improved_rankings, rerank_latency = rerank_all(retrieval_rankings)
    save_json_atomic(rerank_path, {"rankings": improved_rankings,
                                   "latencies": rerank_latency})
    print("Результаты реранжирования сохранены:", rerank_path)
rankings_runtime_version = CHECKPOINT_VERSION
improved_metrics = metrics_for_rankings(improved_rankings)
summary = pd.DataFrame({
    "version": ["dense baseline", "routed hybrid", "hybrid + cross-encoder"],
    "precision@5": [baseline_metrics["precision@5"].mean(),
                      retrieval_metrics["precision@5"].mean(),
                      improved_metrics["precision@5"].mean()],
    "recall@5": [baseline_metrics["recall@5"].mean(),
                   retrieval_metrics["recall@5"].mean(),
                   improved_metrics["recall@5"].mean()],
    "MRR": [baseline_metrics.mrr.mean(), retrieval_metrics.mrr.mean(),
            improved_metrics.mrr.mean()]})
display(summary)
display(improved_metrics.groupby("kind")[["precision@5", "recall@5", "mrr"]].mean())
print("Средняя добавочная задержка реранкера, мс:", round(np.mean(rerank_latency)*1000, 1))

errors = improved_metrics.nsmallest(8, ["recall@5", "mrr"])
display(errors[["kind", "query", "recall@5", "mrr"]])

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6648.71it/s]


Реранкер запущен на cpu: медленнее, но без переполнения Metal-памяти.
Результаты реранжирования загружены из чекпоинта: filmoskop_checkpoints/reranking_518459a61e887960.json


,version,precision@5,recall@5,MRR
0,dense baseline,0.200000,0.758929,0.655288
1,routed hybrid,0.203571,0.767857,0.652907
2,hybrid + cross-encoder,0.250000,0.946429,0.829613


,precision@5,recall@5,mrr
kind,,,
exact,0.213333,1.000,1.000000
multi,0.350000,0.875,0.843750
semantic,0.208000,0.960,0.718333


Средняя добавочная задержка реранкера, мс: 2143.2


,kind,query,recall@5,mrr
23,semantic,скачанная серия перестала открываться без инте...,0.0,0.125000
48,multi,"скачал сериал в дорогу, а место на телефоне ко...",0.5,0.500000
50,multi,сменил пароль — нужно ли что-то делать с устро...,0.5,0.500000
53,multi,заплатил за новинку отдельно — сколько она у м...,0.5,0.500000
54,multi,"у меня телевизор AM-9, приложение не ставится ...",0.5,1.000000
18,semantic,"как сделать, чтобы деньги сами не снимались ка...",1.0,0.333333
20,semantic,"не пускает в аккаунт, пароль вылетел из головы",1.0,0.333333
30,semantic,"боюсь, что ребёнок случайно купит фильм с моей...",1.0,0.333333


Цена улучшения — вторая модель (~0.6B параметров), хранение её весов и один
forward pass на 20 пар на запрос. Это оправдано для прототипа, но latency надо
измерять на целевом железе. Если прирост recall@5 не устойчив, production-вариант
должен оставить E5 и добавить дешёвую маршрутизацию exact-запросов в BM25.

## Этап 5. Генерация ответов

Генерация выполняется через NVIDIA API. Ключ вводится скрыто в следующей
ячейке и хранится только в памяти ядра. Ответы создаёт
`nvidia/nemotron-3.5-lightning-30b-a3b`, а независимый LLM-судья использует
`openai/gpt-oss-20b`. Эмбеддинги, поиск и реранкер остаются локальными.
Температура равна нулю, режим длинных рассуждений отключён.

In [58]:
from getpass import getpass
from openai import OpenAI

nvidia_api_key = getpass("Введите NVIDIA API key: ")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_api_key,
)

completion = client.chat.completions.create(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    messages=[
        {
            "role": "user",
            "content": "Кратко расскажи о преимуществах GPU-вычислений.",
        }
    ],
    temperature=0,
    max_tokens=512,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False,
        }
    },
)

print(completion.choices[0].message.content)

GPU-вычисления (General-Purpose computing on GPU) предлагают несколько ключевых преимуществ:

1.  **Параллельная обработка:** В отличие от CPU (центрального процессора), который обрабатывает задачи последовательно (по одной), GPU содержит тысячи мелких ядер, способных выполнять множество операций одновременно.
2.  **Высокая производительность:** Благодаря параллелизму GPU способны выполнять тяжелые вычисления (матричные операции, графику, физику) в десятки и сотни раз быстрее CPU для определенных задач.
3.  **Энергоэффективность:** Для решения одной и той же задачи GPU потребляет меньше энергии, чем многоядерный процессор, что важно для больших данных и облачных сервисов.
4.  **Масштабируемость:** Возможность соединять несколько видеокарт для еще большего прироста производительности.

**Где это применяется:**
*   Искусственный интеллект и машинное обучение (обучение нейросетей).
*   Криптовалюта-майн잉.
*   Рендеринг графики и видео.
*   Научные симуляции и расчеты.


In [59]:
from openai import APIConnectionError, OpenAI

RAG_BASE_URL = "https://integrate.api.nvidia.com/v1"
GEN_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
JUDGE_MODEL = "openai/gpt-oss-20b"
LLM_EXTRA_BODY = {"chat_template_kwargs": {"enable_thinking": False}}

def check_llm_server():
    "Проверяет доступность OpenAI-compatible сервера и даёт понятную ошибку."
    try:
        client.models.list()
    except APIConnectionError as exc:
        raise RuntimeError(
            f"NVIDIA API недоступен по адресу {RAG_BASE_URL}. "
            "Проверьте интернет-соединение и повторите ячейку."
        ) from exc

check_llm_server()

def retrieve(question, k=TOP_K):
    "Routed hybrid top-20 + cross-encoder; возвращает top-k разделов."
    qv = embedder.encode("query: " + question, normalize_embeddings=True)
    candidates = hybrid_candidates(question, qv, 20)
    scores = reranker.predict([(question, c["text"]) for c in candidates],
                              batch_size=16, show_progress_bar=False)
    return [{**candidates[i], "rerank_score": float(scores[i])}
            for i in np.argsort(-np.asarray(scores))[:k]]

def should_refuse(question, contexts):
    "Route-specific отказ по исходному dense score, описанный в этапе 3."
    if not contexts:
        return True
    threshold = ROUTE_THRESHOLDS.get(query_route(question), THRESHOLD)
    return max(c.get("score", -1.0) for c in contexts) < threshold

def answer(question, contexts=None):
    "Генерирует заземлённый ответ или согласованный отказ."
    contexts = retrieve(question) if contexts is None else contexts
    if should_refuse(question, contexts):
        return REFUSAL
    block = "\n\n".join(f"[Раздел {c['doc_id']}: {c['title']}]\n{c['text']}"
                         for c in contexts)
    prompt = f'''Ты ассистент поддержки онлайн-кинотеатра «Фильмоскоп».
Отвечай кратко и только по КОНТЕКСТУ. Не добавляй тарифы, условия или шаги,
которых там нет. Если сведений недостаточно, ответь дословно: {REFUSAL}

КОНТЕКСТ:
{block}

ВОПРОС: {question}
ОТВЕТ:'''
    response = client.chat.completions.create(
        model=GEN_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0, max_tokens=256, extra_body=LLM_EXTRA_BODY)
    return response.choices[0].message.content.strip()

# Проверка трёх типов и одного gap-запроса
for kind in ["exact", "semantic", "multi", "gap"]:
    q = next(x for x in queries if x["kind"] == kind)
    print(kind, "|", q["query"], "\n", answer(q["query"]), "\n")

exact | что означает ошибка P-102 
 Ошибка P-102: превышен лимит просмотров. Плеер показывает это сообщение, когда число одновременных просмотров на аккаунте превышает лимит тарифа. Просмотр при этом прерывается. Перезапустите приложение и попробуйте снова. Проверьте, что установлена последняя версия приложения и на устройстве есть свободное место. Остановите просмотр на другом устройстве: сколько экранов можно занять одновременно, указано в описании тарифов. 

semantic | вечером фильмы постоянно тормозят и крутится кружок загрузки, что делать 
 В базе знаний нет информации для ответа на этот вопрос. 

multi | чем Стандарт отличается от Максимума и потеряю ли оплаченные дни при переходе 
 В базе знаний нет информации для ответа на этот вопрос. 

gap | есть ли подарочные сертификаты 
 В базе знаний нет информации для ответа на этот вопрос. 



## Эксперимент retrieval: Query Expansion + HyDE

Отдельный эксперимент: три перефразировки и HyDE-документ для улучшения поиска. Baseline не изменяется.

In [ ]:
EXPANSION_MODEL = "openai/gpt-oss-20b"
expansion_key = cache_key(CHECKPOINT_VERSION, EXPANSION_MODEL, queries)
expansion_path = CHECKPOINT_DIR / f"query_expansion_hyde_{expansion_key}.json"
expansion_cache = (json.loads(expansion_path.read_text(encoding="utf-8"))
                   if expansion_path.exists() else {})
print(f"Готовых расширений: {len(expansion_cache)} из 56")

def expand_query_and_hyde(q):
    prompt = f'''Создай ровно 3 короткие поисковые перефразировки русского
вопроса, сохраняя коды, числа и названия. Затем напиши краткий
гипотетический фрагмент справки с возможным ответом. Верни только JSON:
{{"expansions": ["...", "...", "..."], "hypothetical_document": "..."}}
ВОПРОС: {q["query"]}'''
    last_error = None
    for attempt in range(1, 4):
        response = client.chat.completions.create(
            model=EXPANSION_MODEL, messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=4096, reasoning_effort="low",
            response_format={"type": "json_object"})
        raw = response.choices[0].message.content or ""
        try:
            value = json.loads(raw)
            expansions = value.get("expansions", [])
            hypothetical = value.get("hypothetical_document")
            if len(expansions) != 3 or not hypothetical:
                raise ValueError("ожидались 3 expansions и hypothetical_document")
            return value
        except (json.JSONDecodeError, TypeError, ValueError) as exc:
            last_error = exc
            print(f"Некорректный JSON, повтор {attempt}/3: {exc}")
    print("Используем безопасный fallback после трёх неудачных ответов API.")
    question = q["query"].strip()
    return {"expansions": [question, "инструкция: " + question,
                            "раздел справки: " + question],
            "hypothetical_document": question, "fallback": True}

for query_no, q in enumerate(queries, 1):
    if q["kind"] == "gap" or q["query"] in expansion_cache: continue
    expansion_cache[q["query"]] = expand_query_and_hyde(q)
    save_json_atomic(expansion_path, expansion_cache)
    print(f"Expansion сохранён {query_no}/{len(queries)}: {q['query'][:65]}")

expanded_dense_rankings = []
for q, original_qv in zip(queries, query_vectors):
    if q["kind"] == "gap":
        scores = original_qv @ doc_vectors.T
    else:
        item = expansion_cache[q["query"]]
        variants = [q["query"], *item["expansions"]]
        variant_vectors = embedder.encode(["query: " + x for x in variants], normalize_embeddings=True)
        expansion_scores = np.max(variant_vectors @ doc_vectors.T, axis=0)
        hyde_vector = embedder.encode("passage: " + item["hypothetical_document"], normalize_embeddings=True)
        scores = .75 * expansion_scores + .25 * (hyde_vector @ doc_vectors.T)
    expanded_dense_rankings.append(unique_doc_ranking(scores, chunks, 20))

expanded_key = cache_key(CHECKPOINT_VERSION, RERANK_MODEL, expanded_dense_rankings)
expanded_path = CHECKPOINT_DIR / f"reranking_expansion_hyde_{expanded_key}.json"
if expanded_path.exists():
    expanded_rankings = json.loads(expanded_path.read_text(encoding="utf-8"))
else:
    expanded_rankings, _ = rerank_all(expanded_dense_rankings)
    save_json_atomic(expanded_path, expanded_rankings)
expanded_metrics = metrics_for_rankings(expanded_rankings)
fallback_count = sum(bool(x.get("fallback")) for x in expansion_cache.values())
print(f"Fallback использован для {fallback_count} из {len(expansion_cache)} запросов")
cols = ["recall@1", "recall@3", "recall@5", "recall@10", "recall@20", "mrr"]
display(pd.DataFrame({"baseline": baseline_metrics[cols].mean(),
                      "reranker": improved_metrics[cols].mean(),
                      "Query Expansion + HyDE": expanded_metrics[cols].mean()}))
display(expanded_metrics.groupby("kind")[["recall@5", "mrr"]].mean())

Готовых расширений: 0 из 56
Expansion сохранён 1/66: что означает ошибка P-102
Expansion сохранён 2/66: какая версия webOS или Tizen нужна для приложения на телевизоре
Expansion сохранён 3/66: обязателен ли HDCP 2.2 для просмотра в 4K
Expansion сохранён 4/66: как подключить саундбар по HDMI ARC
Expansion сохранён 5/66: что означает ошибка P-109
Expansion сохранён 6/66: что значит ошибка P-111
Expansion сохранён 7/66: сколько стоит тариф Максимум и что в него входит
Expansion сохранён 8/66: сколько дней длится пробный период
Expansion сохранён 9/66: какая скорость интернета нужна для 4K
Expansion сохранён 10/66: есть ли приложение Фильмоскопа для Nintendo Switch
Expansion сохранён 11/66: сколько профилей можно создать в одном аккаунте
Expansion сохранён 12/66: на каком тарифе доступен звук 5.1
Expansion сохранён 13/66: сколько устройств можно привязать к аккаунту
Некорректный JSON, повтор 1/3: Unterminated string starting at: line 1 column 191 (char 190)
Expansion сохранён 14/66: скольк

### Ablation: Query Expansion и HyDE по отдельности

Проверяем каждый метод независимо на одних и тех же сохранённых данных. Новые обращения к LLM не выполняются.

In [ ]:
def build_ablation_dense(mode):
    rankings = []
    for q, original_qv in zip(queries, query_vectors):
        if q["kind"] == "gap":
            scores = original_qv @ doc_vectors.T
        else:
            item = expansion_cache[q["query"]]
            if mode == "query_expansion":
                variants = [q["query"], *item["expansions"]]
                vectors = embedder.encode(["query: " + x for x in variants],
                                          normalize_embeddings=True)
                scores = np.max(vectors @ doc_vectors.T, axis=0)
            elif mode == "hyde":
                vector = embedder.encode("passage: " + item["hypothetical_document"],
                                         normalize_embeddings=True)
                scores = vector @ doc_vectors.T
            else:
                raise ValueError(mode)
        rankings.append(unique_doc_ranking(scores, chunks, 20))
    return rankings

def cached_ablation_rerank(mode):
    dense = build_ablation_dense(mode)
    key = cache_key(CHECKPOINT_VERSION, mode, RERANK_MODEL, dense)
    path = CHECKPOINT_DIR / f"reranking_{mode}_{key}.json"
    if path.exists():
        print(mode, "загружен из чекпоинта")
        return json.loads(path.read_text(encoding="utf-8"))
    rankings, _ = rerank_all(dense)
    save_json_atomic(path, rankings)
    return rankings

query_expansion_rankings = cached_ablation_rerank("query_expansion")
hyde_only_rankings = cached_ablation_rerank("hyde")
query_expansion_metrics = metrics_for_rankings(query_expansion_rankings)
hyde_only_metrics = metrics_for_rankings(hyde_only_rankings)
cols = ["recall@1", "recall@3", "recall@5", "recall@10", "recall@20", "mrr"]
ablation_table = pd.DataFrame({
    "dense baseline": baseline_metrics[cols].mean(),
    "reranker": improved_metrics[cols].mean(),
    "Query Expansion only": query_expansion_metrics[cols].mean(),
    "HyDE only": hyde_only_metrics[cols].mean(),
    "Query Expansion + HyDE": expanded_metrics[cols].mean(),
})
by_kind_table = pd.concat({
    "Query Expansion": query_expansion_metrics.groupby("kind")[["recall@5", "mrr"]].mean(),
    "HyDE": hyde_only_metrics.groupby("kind")[["recall@5", "mrr"]].mean(),
}, axis=1)
ablation_results_path = CHECKPOINT_DIR / "retrieval_ablation_results.json"
save_json_atomic(ablation_results_path, {
    "overall": json.loads(ablation_table.to_json()),
    "by_kind": json.loads(by_kind_table.to_json()),
})
print("Сравнение сохранено:", ablation_results_path)
display(ablation_table)
display(by_kind_table)

Сравнение сохранено: filmoskop_checkpoints/retrieval_ablation_results.json


,dense baseline,reranker,Query Expansion only,HyDE only,Query Expansion + HyDE
recall@1,0.232143,0.446429,0.428571,0.392857,0.419643
recall@3,0.508929,0.758929,0.767857,0.714286,0.741071
recall@5,0.714286,0.866071,0.866071,0.767857,0.848214
recall@10,0.812500,0.919643,0.919643,0.812500,0.901786
recall@20,0.928571,0.928571,0.928571,0.812500,0.901786
mrr,0.496595,0.697343,0.688414,0.632759,0.668771


Query Expansion                HyDE          
                recall@5       mrr  recall@5       mrr
kind                                                  
exact             1.0000  0.800000  0.933333  0.722222
multi             0.8125  0.710417  0.750000  0.697917
semantic          0.8200  0.607381  0.680000  0.537381

## Этап 6. Оценка качества и финальная диагностика

LLM-судья получает только вопрос, контекст, ответ и gold. Он возвращает JSON с
тремя бинарными оценками. Для gap дополнительно применяется строгая
детерминированная проверка согласованной фразы отказа. Судья не заменяет ручной
разбор: ниже печатаются худшие случаи вместе с контекстами.

In [ ]:
def judge(question, contexts, generated, gold):
    "Оценивает faithfulness, relevance и correctness в строгом JSON."
    context = "\n\n".join(c["text"] for c in contexts)
    rubric = f'''Оцени ответ RAG. Верни только JSON:
{{"faithfulness": 0 или 1, "answer_relevance": 0 или 1, "correctness": 0 или 1,
  "reason": "краткая причина"}}
faithfulness=1, если все факты ответа подтверждены контекстом.
answer_relevance=1, если ответ прямо отвечает на вопрос.
correctness=1, если передан эталонный факт (дословное совпадение не требуется).
ВОПРОС: {question}
КОНТЕКСТ: {context}
ОТВЕТ: {generated}
ЭТАЛОН: {gold}'''
    response = client.chat.completions.create(
        model=JUDGE_MODEL, messages=[{"role": "user", "content": rubric}],
        temperature=0, max_tokens=4096, reasoning_effort="low",
        response_format={"type": "json_object"})
    message = response.choices[0].message
    raw = message.content or getattr(message, "reasoning_content", None) or ""
    if not raw.strip():
        raise RuntimeError("LLM-судья вернул пустой ответ; повторите запрос.")
    match = re.search(r"\{.*?\}", raw, re.S)
    if not match:
        raise ValueError("Судья не вернул JSON. Начало ответа: " + raw[:500])
    return json.loads(match.group())

def gold_contexts(q):
    "Возвращает очищенные золотые разделы для oracle-прогона."
    wanted = set(q["relevant_docs"])
    return [{"doc_id": d["doc_id"], "title": d["title"], "text": d["text"],
             "score": 1.0} for d in docs if d["doc_id"] in wanted]

evaluation_key = cache_key(CHECKPOINT_VERSION, GEN_MODEL, JUDGE_MODEL, RAG_BASE_URL, TOP_K,
                           queries, improved_rankings, REFUSAL)
evaluation_path = CHECKPOINT_DIR / f"evaluation_{evaluation_key}.json"
if evaluation_path.exists():
    evaluation = json.loads(evaluation_path.read_text(encoding="utf-8"))
    print(f"Продолжаем оценку: готово {len(evaluation)} из {len(queries)} запросов")
else:
    evaluation = []

completed_queries = {row["query"] for row in evaluation}
for query_no, (q, ranking) in enumerate(zip(queries, improved_rankings), start=1):
    if q["query"] in completed_queries:
        continue
    ctx = ranking[:TOP_K]
    generated = answer(q["query"], ctx)
    scores = judge(q["query"], ctx, generated, q["gold"])
    if q["kind"] == "gap":
        scores["correctness"] = int(generated == REFUSAL)
    oracle_ctx = gold_contexts(q)
    oracle_answer = REFUSAL if q["kind"] == "gap" else answer(q["query"], oracle_ctx)
    oracle_scores = judge(q["query"], oracle_ctx, oracle_answer, q["gold"])
    evaluation.append({"kind": q["kind"], "query": q["query"],
                       "answer": generated, **scores,
                       "oracle_correctness": oracle_scores["correctness"]})
    save_json_atomic(evaluation_path, evaluation)
    print(f"Сохранён запрос {query_no}/{len(queries)}: {q['query'][:70]}")

eval_df = pd.DataFrame(evaluation)
display(eval_df[["faithfulness", "answer_relevance", "correctness",
                 "oracle_correctness"]].mean().to_frame("all"))
display(eval_df.groupby("kind")[["faithfulness", "answer_relevance",
                                 "correctness", "oracle_correctness"]].mean())

Продолжаем оценку: готово 3 из 66 запросов
Сохранён запрос 4/66: как подключить саундбар по HDMI ARC
Сохранён запрос 5/66: что означает ошибка P-109
Сохранён запрос 6/66: что значит ошибка P-111
Сохранён запрос 7/66: сколько стоит тариф Максимум и что в него входит
Сохранён запрос 8/66: сколько дней длится пробный период
Сохранён запрос 9/66: какая скорость интернета нужна для 4K
Сохранён запрос 10/66: есть ли приложение Фильмоскопа для Nintendo Switch
Сохранён запрос 11/66: сколько профилей можно создать в одном аккаунте
Сохранён запрос 12/66: на каком тарифе доступен звук 5.1
Сохранён запрос 13/66: сколько устройств можно привязать к аккаунту
Сохранён запрос 14/66: сколько загрузок можно хранить на одном устройстве
Сохранён запрос 15/66: поддерживается ли телевизор QX-450
Сохранён запрос 16/66: вечером фильмы постоянно тормозят и крутится кружок загрузки, что дела
Сохранён запрос 17/66: включаю кино — на экране всё идёт, а из колонок тишина
Сохранён запрос 18/66: телевизор показывает

,all
faithfulness,0.500000
answer_relevance,0.530303
correctness,0.500000
oracle_correctness,0.515152


,faithfulness,answer_relevance,correctness,oracle_correctness
kind,,,,
exact,0.800,0.866667,0.8000,0.866667
gap,0.900,0.900000,1.0000,1.000000
multi,0.375,0.437500,0.3125,0.250000
semantic,0.240,0.240000,0.2400,0.280000


## A/B-тест генератора: GPT-OSS 20B

Этот блок не изменяет результаты Nemotron. Он использует те же retrieval-результаты
и того же LLM-судью, но сохраняет ответы GPT-OSS в отдельный чекпоинт.

In [ ]:
GPT_OSS_MODEL = "openai/gpt-oss-20b"

def answer_gpt_oss(question, contexts=None):
    "Генерирует ответ GPT-OSS, не меняя глобальную модель Nemotron."
    contexts = retrieve(question) if contexts is None else contexts
    if should_refuse(question, contexts):
        return REFUSAL
    block = "\n\n".join(f"[Раздел {c['doc_id']}: {c['title']}]\n{c['text']}"
                         for c in contexts)
    prompt = f'''Ты ассистент поддержки онлайн-кинотеатра «Фильмоскоп».
Ответь на русском языке кратко и только по КОНТЕКСТУ. Используй все нужные
разделы, если вопрос состоит из нескольких частей. Не добавляй сведения,
которых нет в контексте. Если контекст действительно не содержит ответа,
ответь дословно: {REFUSAL}

КОНТЕКСТ:
{block}

ВОПРОС: {question}
ОТВЕТ:'''
    response = client.chat.completions.create(
        model=GPT_OSS_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0, max_tokens=2048, reasoning_effort="low")
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("GPT-OSS не успел сформировать ответ; повторите запрос.")
    return content.strip()

gpt_oss_key = cache_key(CHECKPOINT_VERSION, GPT_OSS_MODEL, JUDGE_MODEL,
                        RAG_BASE_URL, TOP_K, queries, improved_rankings, REFUSAL)
gpt_oss_path = CHECKPOINT_DIR / f"evaluation_gpt_oss_{gpt_oss_key}.json"
if gpt_oss_path.exists():
    gpt_oss_evaluation = json.loads(gpt_oss_path.read_text(encoding="utf-8"))
    print(f"Продолжаем GPT-OSS: готово {len(gpt_oss_evaluation)} из {len(queries)}")
else:
    gpt_oss_evaluation = []

gpt_oss_completed = {row["query"] for row in gpt_oss_evaluation}
for query_no, (q, ranking) in enumerate(zip(queries, improved_rankings), start=1):
    if q["query"] in gpt_oss_completed:
        continue
    ctx = ranking[:TOP_K]
    generated = answer_gpt_oss(q["query"], ctx)
    scores = judge(q["query"], ctx, generated, q["gold"])
    if q["kind"] == "gap":
        scores["correctness"] = int(generated == REFUSAL)
    oracle_ctx = gold_contexts(q)
    oracle_answer = REFUSAL if q["kind"] == "gap" else answer_gpt_oss(q["query"], oracle_ctx)
    oracle_scores = judge(q["query"], oracle_ctx, oracle_answer, q["gold"])
    gpt_oss_evaluation.append({"kind": q["kind"], "query": q["query"],
                               "answer": generated, **scores,
                               "oracle_correctness": oracle_scores["correctness"]})
    save_json_atomic(gpt_oss_path, gpt_oss_evaluation)
    print(f"GPT-OSS сохранён {query_no}/{len(queries)}: {q['query'][:65]}")

gpt_oss_eval_df = pd.DataFrame(gpt_oss_evaluation)
gpt_oss_runtime_version = CHECKPOINT_VERSION
metric_columns = ["faithfulness", "answer_relevance", "correctness", "oracle_correctness"]
display(pd.DataFrame({
    "Nemotron": eval_df[metric_columns].mean(),
    "GPT-OSS 20B": gpt_oss_eval_df[metric_columns].mean(),
}))
display(gpt_oss_eval_df.groupby("kind")[metric_columns].mean())

GPT-OSS сохранён 1/66: что означает ошибка P-102
GPT-OSS сохранён 2/66: какая версия webOS или Tizen нужна для приложения на телевизоре
GPT-OSS сохранён 3/66: обязателен ли HDCP 2.2 для просмотра в 4K
GPT-OSS сохранён 4/66: как подключить саундбар по HDMI ARC
GPT-OSS сохранён 5/66: что означает ошибка P-109
GPT-OSS сохранён 6/66: что значит ошибка P-111
GPT-OSS сохранён 7/66: сколько стоит тариф Максимум и что в него входит
GPT-OSS сохранён 8/66: сколько дней длится пробный период
GPT-OSS сохранён 9/66: какая скорость интернета нужна для 4K
GPT-OSS сохранён 10/66: есть ли приложение Фильмоскопа для Nintendo Switch
GPT-OSS сохранён 11/66: сколько профилей можно создать в одном аккаунте
GPT-OSS сохранён 12/66: на каком тарифе доступен звук 5.1
GPT-OSS сохранён 13/66: сколько устройств можно привязать к аккаунту
GPT-OSS сохранён 14/66: сколько загрузок можно хранить на одном устройстве
GPT-OSS сохранён 15/66: поддерживается ли телевизор QX-450
GPT-OSS сохранён 16/66: вечером фильмы постоя

,Nemotron,GPT-OSS 20B
faithfulness,0.500000,0.803030
answer_relevance,0.530303,0.909091
correctness,0.500000,0.787879
oracle_correctness,0.515152,0.909091


,faithfulness,answer_relevance,correctness,oracle_correctness
kind,,,,
exact,0.933333,0.933333,0.866667,0.933333
gap,0.900000,0.900000,0.900000,1.000000
multi,0.812500,1.000000,0.750000,0.875000
semantic,0.680000,0.840000,0.720000,0.880000


In [ ]:
# Ручная диагностика худших запросов для лучшего доступного генератора
if CHECKPOINT_VERSION < 3:
    print("ПРЕДУПРЕЖДЕНИЕ: в памяти старая версия очистки корпуса.")
if globals().get("rankings_runtime_version") != CHECKPOINT_VERSION:
    print("ПРЕДУПРЕЖДЕНИЕ: показаны последние доступные rankings до повторного прогона v2.")
if globals().get("gpt_oss_runtime_version") != CHECKPOINT_VERSION:
    print("ПРЕДУПРЕЖДЕНИЕ: показана последняя сохранённая оценка GPT-OSS до повторного прогона v2.")
if "gpt_oss_eval_df" in globals() and len(gpt_oss_eval_df) == len(queries):
    diagnostic_df, diagnostic_model = gpt_oss_eval_df, "GPT-OSS 20B"
else:
    diagnostic_df, diagnostic_model = eval_df, "Nemotron"
print("Диагностируемая модель:", diagnostic_model)
worst = diagnostic_df.sort_values(["correctness", "faithfulness", "answer_relevance"]).head(8)
for _, row in worst.iterrows():
    qi = next(i for i, q in enumerate(queries) if q["query"] == row["query"])
    print("\nВОПРОС:", row["query"])
    print("ОТВЕТ:", row["answer"])
    print("КОНТЕКСТЫ:")
    for c in improved_rankings[qi][:TOP_K]:
        readable_text = repair_mojibake(c["text"])
        print(f"  doc={c['doc_id']} score={c.get('rerank_score', 0):.3f} | {readable_text[:240]}")

target_recall = improved_metrics["recall@5"].mean() >= .85
target_correctness = diagnostic_df["correctness"].mean() >= .85
gap_ok = (diagnostic_df.loc[diagnostic_df.kind == "gap", "answer"] == REFUSAL).all()
print("\nЦЕЛИ: recall@5", target_recall,
      "| correctness", target_correctness, "| gap refusal", gap_ok)

ПРЕДУПРЕЖДЕНИЕ: показаны последние доступные rankings до повторного прогона v2.
ПРЕДУПРЕЖДЕНИЕ: показана последняя сохранённая оценка GPT-OSS до повторного прогона v2.
Диагностируемая модель: GPT-OSS 20B

ВОПРОС: есть ли приложение Фильмоскопа для Nintendo Switch
ОТВЕТ: В базе знаний нет информации для ответа на этот вопрос.
КОНТЕКСТЫ:
  doc=133 score=1.000 | Игровые консоли
Приложение «Фильмоскоп» доступно на игровых консолях PlayStation 5 и Xbox Series: установите его из магазина приложений консоли и войдите в аккаунт по коду.
Для Nintendo Switch приложения нет, и его выпуск пока не планируетс
  doc=150 score=0.894 | Игры и кино на одной консоли
Игры и кино на одной консоли
У вас уже есть всё для домашнего кинотеатра — просто вы называли это игровой консолью.
Большой экран, нормальный звук, быстрый интернет и приложение «Фильмоскоп», которое ставится из
  doc=86 score=0.491 | Поддерживаемые устройства
Поддерживаемые устройства
На каких устройствах работает "Фильмоскоп"
Телевизоры Sma

## Итоговый вывод и финальная конфигурация

В работе построен полный прототип RAG-ассистента для базы знаний онлайн-кинотеатра «Фильмоскоп»: выполнены очистка HTML и исправление mojibake, разбиение документов на смысловые фрагменты, построение эмбеддингов, поиск, реранжирование, генерация ответа, обработка запросов без ответа и автоматическая оценка качества. Промежуточные результаты сохраняются в чекпоинты, поэтому длительные этапы можно продолжать после остановки.

### Что сравнивалось

- dense retrieval по целому документу и paragraph-aware retrieval;
- dense baseline и cross-encoder reranker;
- обычный поиск, Query Expansion, HyDE и их комбинация;
- два генератора: Nemotron 3.5 Lightning 30B-A3B и GPT-OSS 20B;
- обычный RAG-прогон и oracle-прогон с эталонными документами.

### Основные результаты

После ревью корпус был дополнительно очищен: исключены 21 архивный, 13 служебных и 7 дублирующих документов; ни один эталонно-релевантный документ не удалён. На очищенном корпусе routed hybrid + cross-encoder достиг `recall@5 = 0.9464` и `MRR = 0.8296`. По типам запросов recall@5 составил 1.00 для exact, 0.96 для semantic и 0.875 для multi. Dense baseline после очистки получил `recall@5 = 0.7589`, `MRR = 0.6553`, а routed BM25+dense до реранкера — `0.7679` и `0.6529`.

В исходном A/B-тесте GPT-OSS 20B существенно превзошёл Nemotron: `faithfulness = 0.8030`, `answer relevance = 0.9091`, `correctness = 0.7879`, `oracle correctness = 0.9091`. Поэтому GPT-OSS 20B выбран финальным генератором. Эти generation-метрики относятся к предыдущей версии корпуса; после изменения состава документов они пересчитываются отдельным v3-чекпойнтом и не смешиваются со старыми результатами.

### Финальные параметры

| Компонент | Выбранное значение |
|---|---|
| Embeddings | `intfloat/multilingual-e5-base` |
| Индексация | paragraph-aware chunks, нормализованные embeddings |
| Кандидаты | routed BM25+dense через RRF для exact; dense для остальных; top-20 |
| Реранкер | `BAAI/bge-reranker-v2-m3`, CPU, batch size 16 |
| Контекст генератора | top-5 после реранжирования |
| Генератор | `openai/gpt-oss-20b` через NVIDIA API |
| Параметры генерации | `temperature=0`, `max_tokens=2048`, `reasoning_effort=low` |
| LLM-судья | `openai/gpt-oss-20b`, JSON output, `temperature=0` |
| Отказ | фиксированная фраза; отдельный 5-й перцентиль top-1 dense score для exact/semantic/multi |
| Воспроизводимость | `seed=42`, атомарные JSON/NPZ-чекпоинты, `CHECKPOINT_VERSION=3` |

### Финальный вердикт

После очистки корпуса цель по поиску достигнута с запасом (`recall@5 = 0.9464` при цели `≥ 0.85`). Для окончательного вывода по целевой correctness `≥ 0.85` необходимо выполнить v3-ячейки генерации и LLM-судьи с NVIDIA API: старую оценку `0.7879` нельзя выдавать за результат изменённого pipeline. Порог отказа в production также следует калибровать на отдельной validation-выборке.